# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all `RecordSet` entries, then show their associated field and column `@id`s as described in the schema.

In [ ]:
# Get all record set @id's in the dataset
record_sets = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
print('Available record sets:')
for rs_id in record_sets:
    print(f"  RecordSet @id: {rs_id}")

# For each record set, print field and column @id's
if not record_sets:
    print('No explicit record sets found in toplevel metadata; attempting to infer available record sets from croissant files...')
    # Load all available record sets via dataset API (mlc >=0.3)
    try:
        record_set_objs = dataset.record_sets
        for rs in record_set_objs:
            print(f"  RecordSet @id: {rs.id}")
            print(f"    Fields: {[f.id for f in rs.fields]}")
            print(f"    Columns: {[c.id for c in rs.columns]}")
    except Exception as e:
        print('Could not enumerate record sets:', e)
else:
    # show fields and columns for each record set
    for rs_id in record_sets:
        try:
            rs_obj = dataset.record_set(rs_id)
            print(f"  RecordSet @id: {rs_id}")
            print(f"    Fields: {[f.id for f in rs_obj.fields]}")
            print(f"    Columns: {[c.id for c in rs_obj.columns]}")
        except Exception as e:
            print(f"  Could not load record set {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

`mlcroissant` exposes data record sets via their `@id` fields only. We will select one record set and show its contents in a DataFrame.

In [ ]:
# List all record set @id's
from collections.abc import Mapping
record_sets = []
if hasattr(dataset, 'record_sets') and dataset.record_sets:
    record_sets = [rs.id for rs in dataset.record_sets]

if not record_sets:
    # Try to fetch from metadata JSON
    rs_ids = metadata.to_json().get('recordSet', [])
    record_sets = [x['@id'] if isinstance(x, Mapping) else x for x in rs_ids]

print("Available record set IDs:")
for r in record_sets:
    print(f"  {r}")

dataframes = {}
for rs_id in record_sets:
    try:
        print(f"Loading records for {rs_id} ...")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded DataFrame ({df.shape[0]} rows, {df.shape[1]} columns)")
        print(f"  Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# Pick the first record set for exploration
if record_sets:
    main_rs = record_sets[0]
    df = dataframes[main_rs]
    print(f"\nFirst 5 rows of record set {main_rs}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will demonstrate, for a numeric field discovered in the first record set, how to filter, normalize, and group. All field and column reference is via the field's `@id` as required.

In [ ]:
### Identify numeric fields and group fields from columns metadata
if record_sets:
    main_rs_id = record_sets[0]
    rs_obj = None
    # Record set object
    try:
        rs_obj = dataset.record_set(main_rs_id)
    except:
        rs_obj = None
    
    numeric_field_id = None
    group_field_id = None
    if rs_obj is not None:
        columns = rs_obj.columns
        numeric_candidates = [c.id for c in columns if getattr(c, 'data_type', '').lower() in ['number', 'integer', 'float']]
        print('Numeric candidate columns:', numeric_candidates)
        if numeric_candidates:
            numeric_field_id = numeric_candidates[0]
        # As group field, use any non-numeric text column
        text_candidates = [c.id for c in columns if getattr(c, 'data_type', '').lower() in ['string', 'text']]
        if text_candidates:
            group_field_id = text_candidates[0]
        print(f"Field selected for analysis: numeric @id={numeric_field_id}, group @id={group_field_id}")
    else:
        # Fallback: use column names (first float/int for numeric, first str for group)
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]):
                group_field_id = col
                break
        print(f"[Fallback] Field selected for analysis: numeric @id={numeric_field_id}, group @id={group_field_id}")

    # Start EDA if possible
    if numeric_field_id is not None and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        mean_val = filtered_df[numeric_field_id].mean()
        std_val = filtered_df[numeric_field_id].std()
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - mean_val) / (std_val if std_val else 1)
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (showing first 5 groups):")
            display(grouped_df.head())
    else:
        print('Could not determine a suitable numeric field for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field, and a boxplot of normalized values grouped by the group field (when available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id in df.columns:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    # Histogram
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=ax[0])
    ax[0].set_title(f'Distribution of {numeric_field_id}')
    ax[0].set_xlabel(numeric_field_id)

    # Normalized boxplot by group
    if group_field_id and group_field_id in df.columns:
        # Ensure a normalized column exists
        col = f"{numeric_field_id}_normalized"
        if col not in df.columns or df[col].isna().all():
            mean_val = df[numeric_field_id].mean()
            std_val = df[numeric_field_id].std() or 1
            df[col] = (df[numeric_field_id] - mean_val) / std_val
        sns.boxplot(x=group_field_id, y=col, data=df, ax=ax[1])
        ax[1].set_title(f'Normalized {numeric_field_id} by {group_field_id}')
        ax[1].set_ylabel(f'{numeric_field_id}_normalized')
        ax[1].set_xlabel(group_field_id)
        plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No suitable numeric field/data for plotting.')

## 6. Conclusion
In this notebook, we used the FAIR^2 dataset's Croissant schema to programmatically access and explore its record sets, loading the main table, filtering and normalizing a selected numeric field, and visualizing important distributions.

Key field, column, and record set references are always by their `@id`, supporting reproducible code.

Next steps: investigate additional record sets, join across them (if multiple are present), and adapt the EDA/visualization steps to domain-specific questions (e.g. identifying biomarkers, correlations between measured protein properties, etc).